'''
Autogen
AutoGen is an open-source framework for building collaborative, multi‑agent AI systems that use LLM‑driven agents to communicate and solve complex tasks autonomously.

From an architectural perspective, AutoGen agents themselves are built around:
User Proxy Agent — Represents the human or system initiating a request
Assistant Agent — The main reasoning and problem‑solving agent.
Tool/Function Agent — Executes specialized operations such as API calls, computation, or data retrieval.

UserProxyAgent (you) -> asks-> AssistantAgent (AI) -> answers LLM (gpt-4o brain)
'''

In [ ]:
# pip install -U autogen autogen-agentchat autogen-core autogen-ext

In [ ]:
import autogen
import os

apikey = os.environ['OPENAI_API_KEY']

In [ ]:
# --------------------------
# Example 1) Getting Started
# --------------------------

# LLM configuration
# Creates a configuration dictionary for the LLM (Large Language Model).
llm_config = {"model": "gpt-4o",  "api_key": apikey, "temperature": 0 }

# Assistant agent
# Creates an AI assistant agent that will answer questions.
assistant = autogen.AssistantAgent(name="assistant", llm_config=llm_config )

# User proxy agent
# Creates a user-side agent that represents the human.
# This agent acts like you (the user) in the conversation.

# code_execution = False: Disabling code execution.
# Agent will:
# NOT run Python code
# ONLY chat with the assistant

user_proxy = autogen.UserProxyAgent(name="user_proxy", code_execution_config=False)

# Start conversation
prompt = "How is Income Tax calculated in India"
user_proxy.initiate_chat(assistant,message=prompt)

In [ ]:
# -------------------------------------
# Example 2) How to enable code execution
# -------------------------------------
# Assistant writes code
# UserProxy executes the code
# Result comes back to the assistant

llm_config = {"model": "gpt-4o", "api_key":apikey, "temperature": 0}

# 🔹 Assistant agent (writes code)
assistant = autogen.AssistantAgent(name="assistant", llm_config=llm_config)

# 🔹 User proxy agent (EXECUTES code)
user_proxy = autogen.UserProxyAgent(name="user_proxy",
    # THIS ENABLES CODE EXECUTION
    code_execution_config={
        "work_dir": "coding_workspace",  # folder where code runs: Check from os.getcwd()
        "use_docker": False,              # set True in production
        "execute_code": True
    }
)

# Start conversation
msg = ''' Write Python code to that will take the birth date as input. 
If the day and month match with today's date, print "Happy Birthday".
Else, print a message "Your birthdays is <days> away.
'''

user_proxy.initiate_chat(assistant, message=msg)

In [ ]:
# ---------------------------
# Example 3) How to Add tools
# ---------------------------

'''
In AutoGen, “tools” are Python functions that an agent can call when needed 
(instead of only replying with text).

There are 2 common patterns:
Register tools on the UserProxy (recommended + simplest)
Register tools directly on the Assistant
'''

# 1) LLM config
llm_config = {"model": "gpt-4o", "api_key": apikey, "temperature": 0 }

# 2) Assistant (decides WHEN to use tools)
assistant = autogen.AssistantAgent(
    name="assistant",
    llm_config=llm_config,
    system_message=( "Use tools when needed. When you have the final answer, "
            "respond with the final answer and then write TERMINATE on a new line."), )

# 3) User proxy (executes the tools)
user_proxy = autogen.UserProxyAgent(
    name="user_proxy",
    human_input_mode="NEVER",   # no manual approval
    code_execution_config=False, # tools will run, not arbitrary code blocks
     is_termination_msg=lambda msg: "TERMINATE" in (msg.get("content") or "")
)


# 4) Define tools (plain Python functions)
# all functions must be properly annotated

def km2miles(km:float) -> float:
  return( km * 0.621371)

def miles2km(miles:float) ->float:
  return(miles / 0.621371)

# 5) Register tools so the assistant can call them

autogen.register_function(
    km2miles,
    caller=assistant,      # the agent that is allowed to CALL the tool
    executor=user_proxy,   # the agent that will EXECUTE the tool
    name="km2miles",
    description="Convert Kilometer to Miles. Input: km")

autogen.register_function(
    miles2km,
    caller=assistant,      # the agent that is allowed to CALL the tool
    executor=user_proxy,   # the agent that will EXECUTE the tool
    name="miles2km",
    description="Convert Miles to Kilometer. Input: miles")

# 2 new things here
# i) store the result in a variable
# ii) Suppress the scrolling output

result = user_proxy.initiate_chat(
    assistant,
    message="A car travels 30 Kms How many miles has it run. Use the tools",
    silent=True
    )

In [ ]:
print(result)

In [10]:
# complete chat output (history)
result.chat_history

# total count of chat history
total_results = len(result.chat_history)
print(total_results)

# # print each conversation output
for i in range(total_results):
  # print(result.chat_history[i]["content"])
  print(f"Content: {result.chat_history[i]['content']}")

4
Content: A car travels 120 miles. How many Kilometers has it run. Use the tools
Content: None
Content: 193.1213397471076
Content: The car has traveled approximately 193.12 kilometers. 

TERMINATE


In [9]:
result2 = user_proxy.initiate_chat(
    assistant,
    message="A car travels 120 miles. How many Kilometers has it run. Use the tools",
    silent=False
    )

user_proxy (to assistant):

A car travels 120 miles. How many Kilometers has it run. Use the tools

--------------------------------------------------------------------------------
assistant (to user_proxy):

***** Suggested tool call (call_Xo4QTUrRYeBV8KKxhzAoR9lB): miles2km *****
Arguments: 
{"miles":120}
*************************************************************************

--------------------------------------------------------------------------------

>>>>>>>> EXECUTING FUNCTION miles2km...
Call ID: call_Xo4QTUrRYeBV8KKxhzAoR9lB
Input arguments: {'miles': 120}

>>>>>>>> EXECUTED FUNCTION miles2km...
Call ID: call_Xo4QTUrRYeBV8KKxhzAoR9lB
Input arguments: {'miles': 120}
Output:
193.1213397471076
user_proxy (to assistant):

***** Response from calling tool (call_Xo4QTUrRYeBV8KKxhzAoR9lB) *****
193.1213397471076
**********************************************************************

--------------------------------------------------------------------------------
assistant (to us

In [ ]:
# ---------------------------------------
# 4) How to make multi-agent conversation
# ----------------------------------------

# 1) LLM config
llm_config = {"model": "gpt-4o", "api_key": apikey, "temperature": 0 }

# 2) Create multiple specialized assistant agents
planner = autogen.AssistantAgent(
    name="planner",
    llm_config=llm_config,
    system_message="You break the problem into steps and delegate."
)

researcher = autogen.AssistantAgent(
    name="researcher",
    llm_config=llm_config,
    system_message="You focus on definitions, examples, and clear explanations."
)

critic = autogen.AssistantAgent(
    name="critic",
    llm_config=llm_config,
    system_message="You check for mistakes, ambiguity, and improve clarity."
)

# 3) User proxy (the human side)
user_proxy = autogen.UserProxyAgent(
    name="user_proxy",
    human_input_mode="NEVER",
    code_execution_config=False,
    # Stop when someone says TERMINATE
    is_termination_msg=lambda msg: "TERMINATE" in (msg.get("content") or "")
)

# 4) Create a group chat
groupchat = autogen.GroupChat(
    agents=[user_proxy, planner, researcher, critic],
    messages=[],
    max_round=3,                 # hard stop to prevent infinite loops
)

# 5) Manager decides turn-taking
manager = autogen.GroupChatManager(
    groupchat=groupchat,
    llm_config=llm_config,
    system_message=(
        "Coordinate the agents. When the final answer is ready, "
        "ask the best agent to provide it and end with TERMINATE."
    ),
)

# 6) Start the multi-agent conversation (user -> manager -> agents)
prompt = "How does Exercise help to maintain a good health"

result = user_proxy.initiate_chat(
    manager,
    message=prompt,
    silent=True
)

In [ ]:
total_results = len(result.chat_history)
print(total_results)

In [ ]:
for i in range(total_results):
  print(f"Content: {result.chat_history[i]["content"]}")

In [ ]:
print(f"Content: {result.chat_history[2]}")

In [ ]:
for i in range(total_results):
    print(result.chat_history[i]['name'])

In [ ]:
'''
 Out of "planner, researcher, critic", which output will be the final ?

The final output is the last message in the group chat
Typically produced by the agent instructed to give the final answer
And usually marked with TERMINATE
'''